# Task 1: Exploratory Data Analysis — Answers

**Data source**: `measurements_clean` table from DuckDB (via DBT pipeline)

**Important**: Q1-Q3 consider only measurements with `instrument_status = 0` (Normal) and exclude missing values (`clean_value IS NOT NULL`, i.e. raw values of -1 are excluded).

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect('../dbt_pollution/dev.duckdb', read_only=True)

# Quick data overview
print("Table: measurements_clean")
print(f"Total rows: {con.sql('SELECT COUNT(*) FROM measurements_clean').fetchone()[0]:,}")
print(f"Normal status rows: {con.sql('SELECT COUNT(*) FROM measurements_clean WHERE instrument_status = 0 AND clean_value IS NOT NULL').fetchone()[0]:,}")
print(f"\nDate range:")
print(con.sql("SELECT MIN(measurement_datetime), MAX(measurement_datetime) FROM measurements_clean").df().to_string(index=False))
print(f"\nStations: {con.sql('SELECT COUNT(DISTINCT station_code) FROM measurements_clean').fetchone()[0]}")
print(f"Pollutants: {con.sql('SELECT COUNT(DISTINCT item_code) FROM measurements_clean').fetchone()[0]}")

Table: measurements_clean
Total rows: 3,729,528
Normal status rows: 3,606,696

Date range:
min(measurement_datetime) max(measurement_datetime)
               2021-01-01       2023-12-31 23:00:00

Stations: 25
Pollutants: 6


## Q1: Average daily SO2 concentration — station average

> Average daily SO2 concentration across all districts over the entire period. Give the station average. Provide the answer with 5 decimals.

**Approach**: 
1. Compute daily average SO2 per station (only Normal status, excluding -1 values)
2. Average all daily values per station → one value per station
3. Average across all stations → grand station average

In [2]:
# Q1: Average daily SO2 — station average
q1_detail = con.sql('''
    WITH daily_avg AS (
        SELECT
            station_code,
            measurement_datetime::DATE AS day,
            AVG(clean_value) AS daily_avg_so2
        FROM measurements_clean
        WHERE item_code = 0  -- SO2
          AND instrument_status = 0  -- Normal only
          AND clean_value IS NOT NULL
        GROUP BY station_code, measurement_datetime::DATE
    ),
    station_avg AS (
        SELECT
            station_code,
            AVG(daily_avg_so2) AS station_avg_so2
        FROM daily_avg
        GROUP BY station_code
    )
    SELECT * FROM station_avg ORDER BY station_code
''').df()

print("Per-station average daily SO2:")
print(q1_detail.to_string(index=False))

answer_q1 = round(q1_detail['station_avg_so2'].mean(), 5)
print(f"\n>>> Q1 ANSWER: {answer_q1}")

Per-station average daily SO2:
 station_code  station_avg_so2
          204         0.004262
          205         0.003584
          206         0.003650
          207         0.004279
          208         0.004340
          209         0.003929
          210         0.004346
          211         0.004495
          212         0.005444
          213         0.005854
          214         0.003626
          215         0.003162
          216         0.004327
          217         0.004586
          218         0.004248
          219         0.005123
          220         0.005220
          221         0.003933
          222         0.004463
          223         0.003741
          224         0.004821
          225         0.004221
          226         0.005294
          227         0.003986
          228         0.004092

>>> Q1 ANSWER: 0.00436


## Q2: Average CO per season at station 209

> Analyse how pollution levels vary by season. Return the average levels of CO per season at the station 209. (Take the whole month of December as part of winter, March as spring, and so on.) Provide the answer with 5 decimals.

In [3]:
# Q2: Average CO per season at station 209
q2 = con.sql('''
    SELECT
        season,
        ROUND(AVG(clean_value), 5) AS avg_co
    FROM measurements_clean
    WHERE item_code = 4  -- CO
      AND station_code = 209
      AND instrument_status = 0
      AND clean_value IS NOT NULL
    GROUP BY season
    ORDER BY
        CASE season
            WHEN 'Spring' THEN 1
            WHEN 'Summer' THEN 2
            WHEN 'Fall' THEN 3
            WHEN 'Winter' THEN 4
        END
''').df()

print(">>> Q2 ANSWER: Average CO per season at station 209:")
print(q2.to_string(index=False))

>>> Q2 ANSWER: Average CO per season at station 209:
season  avg_co
Spring 0.47805
Summer 0.42521
  Fall 0.49979
Winter 0.68040


## Q3: Hour with highest O3 variability (Standard Deviation)

> Which hour presents the highest variability (Standard Deviation) for the pollutant O3? Treat all stations as equal.

**Approach**: Pool all stations together (each has roughly equal coverage) and compute std dev of O3 per hour of day.

In [4]:
# Q3: Hour with highest O3 std dev
q3 = con.sql('''
    SELECT
        hour,
        ROUND(STDDEV(clean_value), 5) AS std_o3,
        ROUND(AVG(clean_value), 5) AS avg_o3,
        COUNT(*) AS n
    FROM measurements_clean
    WHERE item_code = 5  -- O3
      AND instrument_status = 0
      AND clean_value IS NOT NULL
    GROUP BY hour
    ORDER BY hour
''').df()

print("O3 Standard Deviation by hour:")
print(q3.to_string(index=False))

max_row = q3.loc[q3['std_o3'].idxmax()]
print(f"\n>>> Q3 ANSWER: Hour {int(max_row['hour'])} (std = {max_row['std_o3']})")

O3 Standard Deviation by hour:
 hour  std_o3  avg_o3     n
    0 0.01398 0.01849 25186
    1 0.01420 0.01920 25205
    2 0.01420 0.01946 25235
    3 0.01401 0.01916 25199
    4 0.01352 0.01807 25225
    5 0.01238 0.01537 25250
    6 0.01107 0.01261 25302
    7 0.01091 0.01254 25291
    8 0.01183 0.01522 25297
    9 0.01317 0.01938 25250
   10 0.01487 0.02426 25117
   11 0.01717 0.02975 24917
   12 0.01975 0.03520 24788
   13 0.02179 0.03915 24721
   14 0.02328 0.04147 24614
   15 0.02385 0.04190 24786
   16 0.02335 0.03995 24862
   17 0.02204 0.03577 25019
   18 0.02010 0.03044 25225
   19 0.01804 0.02606 25300
   20 0.01632 0.02305 25303
   21 0.01514 0.02099 25249
   22 0.01432 0.01929 25282
   23 0.01399 0.01840 25170

>>> Q3 ANSWER: Hour 15 (std = 0.02385)


## Q4: Station with most "Abnormal data" measurements

> Which is the station code with more measurements labeled as "Abnormal data"?

Note: "Abnormal data" = `instrument_status = 9`

In [5]:
# Q4: Station with most "Abnormal data" (status=9)
q4 = con.sql('''
    SELECT station_code, COUNT(*) AS abnormal_data_count
    FROM measurements_clean
    WHERE instrument_status = 9
    GROUP BY station_code
    ORDER BY abnormal_data_count DESC
''').df()

print("Abnormal data count by station:")
print(q4.to_string(index=False))
print(f"\n>>> Q4 ANSWER: Station {q4.iloc[0]['station_code']} ({q4.iloc[0]['abnormal_data_count']} records)")

Abnormal data count by station:
 station_code  abnormal_data_count
          211                 3197
          214                 1386
          210                 1349
          216                 1288
          222                 1232
          220                 1223
          207                 1112
          225                 1103
          228                 1038
          224                  895
          208                  880
          209                  815
          227                  808
          213                  574
          219                  443
          226                  360
          204                  320
          212                  318
          206                  317
          215                  242
          218                  230
          221                  176
          217                  128
          205                  123
          223                  111

>>> Q4 ANSWER: Station 211 (3197 records)


## Q5: Station with most "not normal" measurements

> Which station code has more "not normal" measurements (!= 0)?

In [6]:
# Q5: Station with most not-normal (status != 0)
q5 = con.sql('''
    SELECT station_code, COUNT(*) AS not_normal_count
    FROM measurements_clean
    WHERE instrument_status != 0
    GROUP BY station_code
    ORDER BY not_normal_count DESC
''').df()

print("Not-normal measurement count by station:")
print(q5.to_string(index=False))
print(f"\n>>> Q5 ANSWER: Station {q5.iloc[0]['station_code']} ({q5.iloc[0]['not_normal_count']} records)")

Not-normal measurement count by station:
 station_code  not_normal_count
          208              9129
          213              7843
          224              7084
          212              7068
          211              6410
          216              5157
          207              4846
          227              4646
          219              4143
          220              3949
          225              3865
          209              3610
          206              3497
          210              3374
          222              3198
          214              3153
          228              2920
          226              2394
          204              2040
          218              1896
          221              1600
          215              1576
          223              1286
          217              1150
          205              1132

>>> Q5 ANSWER: Station 208 (9129 records)


## Q6: Air quality classification counts for PM2.5

> Return the count of `Good`, `Normal`, `Bad` and `Very bad` records for all the station codes of PM2.5 pollutant.

**Thresholds** (from `pollutant_data.csv` for PM2.5, item_code=8):
- Good: value <= 15
- Normal: 15 < value <= 35
- Bad: 35 < value <= 75
- Very bad: value > 75

In [7]:
# Q6: PM2.5 air quality classification counts
q6 = con.sql('''
    SELECT
        air_quality,
        COUNT(*) AS cnt
    FROM measurements_clean
    WHERE item_code = 8  -- PM2.5
      AND air_quality IS NOT NULL
    GROUP BY air_quality
    ORDER BY
        CASE air_quality
            WHEN 'Good' THEN 1
            WHEN 'Normal' THEN 2
            WHEN 'Bad' THEN 3
            WHEN 'Very bad' THEN 4
        END
''').df()

print(">>> Q6 ANSWER: PM2.5 air quality classification counts:")
print(q6.to_string(index=False))
print(f"\nTotal classified records: {q6['cnt'].sum():,}")

>>> Q6 ANSWER: PM2.5 air quality classification counts:
air_quality    cnt
       Good 233717
     Normal 265584
        Bad 101956
   Very bad  16797

Total classified records: 618,054


## Summary of Answers

| Question | Answer |
|----------|--------|
| Q1: Avg daily SO2 (station avg) | **0.00436** |
| Q2: Avg CO at station 209 | Spring: **0.47805**, Summer: **0.42521**, Fall: **0.49979**, Winter: **0.68040** |
| Q3: Hour with highest O3 std dev | **Hour 15** (std = 0.02385) |
| Q4: Station with most "Abnormal data" | **Station 211** (3,197 records) |
| Q5: Station with most "not normal" | **Station 208** (9,129 records) |
| Q6: PM2.5 quality counts | Good: 233,717 / Normal: 265,584 / Bad: 101,956 / Very bad: 16,797 |

In [8]:
con.close()